# Customer Segmentation & Retention Analysis — E-Commerce
## Phase 2: RFM Feature Engineering & Customer Segmentation

**Goal:** Group 5,878 customers into behavioural segments using RFM features and K-Means clustering,  
then translate each cluster into an actionable business strategy.

**Pipeline summary:**
1. Recency / Frequency / Monetary (RFM) table built in `src/segmentation.py`
2. Features log-transformed + standardised to reduce extreme skewness
3. Optimal k selected by silhouette score (range k=2 → k=10)
4. Segments labelled based on actual cluster mean values

---

## Section 1 — Setup & Load RFM Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.1f}'.format)

SEGMENTS_PATH = '../data/processed/rfm_segments.csv'
ELBOW_PLOT    = '../data/processed/elbow_silhouette.png'

print('Libraries loaded.')

In [ ]:
rfm = pd.read_csv(SEGMENTS_PATH, index_col=0)

print(f'Shape         : {rfm.shape}')
print(f'Segments found: {rfm["Segment"].unique().tolist()}')
print(f'Cluster IDs   : {sorted(rfm["Cluster"].unique().tolist())}')
rfm.head()

---
## Section 2 — RFM Distributions

Before evaluating the clusters, it is important to understand the *shape* of each RFM dimension.  
The distributions tell us:
- How recently most customers transacted
- Whether the customer base skews toward one-time or repeat buyers
- How revenue is distributed — evenly, or concentrated in a few high-spend accounts

In [ ]:
# Side-by-side Recency / Frequency / Monetary histograms

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Recency (days since last purchase)',
        'Frequency (unique orders)',
        'Monetary (total spend £)',
    ]
)

# Recency
fig.add_trace(
    go.Histogram(x=rfm['Recency'], nbinsx=50, marker_color='steelblue',
                 name='Recency', showlegend=False),
    row=1, col=1
)

# Frequency — capped at 50 for readability
fig.add_trace(
    go.Histogram(x=rfm['Frequency'].clip(upper=50), nbinsx=50,
                 marker_color='coral', name='Frequency', showlegend=False),
    row=1, col=2
)

# Monetary — log scale
fig.add_trace(
    go.Histogram(x=np.log1p(rfm['Monetary']), nbinsx=50,
                 marker_color='mediumseagreen', name='Monetary (log)', showlegend=False),
    row=1, col=3
)

fig.update_layout(
    title_text='RFM Feature Distributions — 5,878 Customers',
    title_font_size=15,
    height=380,
    template='plotly_white',
)
fig.update_xaxes(title_text='Days', row=1, col=1)
fig.update_xaxes(title_text='Orders (capped at 50)', row=1, col=2)
fig.update_xaxes(title_text='log(1 + £ Spend)', row=1, col=3)
fig.show()

In [ ]:
# Descriptive statistics
print('RFM Percentiles:')
rfm[['Recency','Frequency','Monetary']].quantile([0.25, 0.5, 0.75, 0.90, 0.99])

**Business Insight — What the distributions reveal:**

| Dimension | Key finding | Business implication |
|---|---|---|
| **Recency** | Bimodal — large spike near 0 and another beyond 300 days | Two natural populations: active buyers and lapsed customers |
| **Frequency** | Heavily right-skewed; 75th percentile is only ~7 orders | The majority are low-frequency buyers; a small group of repeat customers drives disproportionate volume |
| **Monetary** | Extreme right skew (raw); log-normal after transformation | A small number of wholesale/B2B accounts dominate revenue — log transform was essential before clustering |

These distributions confirm that **segmentation is necessary**: aggregate averages would be misleading. We need cluster-level profiles to understand who our customers actually are.

---
## Section 3 — Elbow & Silhouette Curves

To choose the number of clusters objectively, we tested k = 2 through 10 using:
- **Elbow method (Inertia):** measures total within-cluster variance — lower is better, but decreases monotonically
- **Silhouette score:** measures how well each point fits its own cluster vs neighbouring clusters — ranges from -1 to +1; higher is better

The silhouette score is the more reliable of the two for selecting k.

In [ ]:
img = mpimg.imread(ELBOW_PLOT)
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

**Business Insight — Why this k was selected:**

The silhouette score peaks at **k = 2**, with a score of **0.439** — well above the scores for k = 3 through 10.  
This tells us the data has two strongly differentiated customer groups:

1. **Active, high-value buyers** — low recency, high frequency, high spend
2. **Dormant / at-risk buyers** — high recency, very low frequency, low spend

While additional clusters could be introduced for more granularity, the data does not naturally support them — forcing more clusters would create overlapping, ambiguous groups that are harder to act on. Two clear segments is the honest, statistically supported answer for this dataset.

> Note: k = 4 is the second-best score (0.365), making it a valid choice if the business requires more segment granularity for campaign targeting.

---
## Section 4 — Cluster Visualisation

Visualising clusters in RFM space confirms that the algorithm found meaningful, well-separated groups — not artefacts of the algorithm.

In [ ]:
# 3D scatter plot — R, F, M on three axes, coloured by Segment

# Cap Monetary for visual clarity (extreme outliers compress the view)
rfm_plot = rfm.copy()
rfm_plot['Monetary_capped'] = rfm_plot['Monetary'].clip(upper=rfm_plot['Monetary'].quantile(0.99))

fig = px.scatter_3d(
    rfm_plot.reset_index(),
    x='Recency', y='Frequency', z='Monetary_capped',
    color='Segment',
    title='Customer Segments in RFM Space (3D)',
    labels={'Monetary_capped': 'Monetary (£, capped 99th pct)'},
    opacity=0.65,
    height=600,
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Bold,
)
fig.update_traces(marker_size=3)
fig.show()

In [ ]:
# 2D scatter plots: R vs F, R vs M, F vs M — coloured by Segment

pairs = [
    ('Recency',   'Frequency',         'Recency vs Frequency'),
    ('Recency',   'Monetary_capped',   'Recency vs Monetary'),
    ('Frequency', 'Monetary_capped',   'Frequency vs Monetary'),
]

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[p[2] for p in pairs],
)

colors = px.colors.qualitative.Bold
segments = rfm_plot['Segment'].unique().tolist()
color_map = {seg: colors[i % len(colors)] for i, seg in enumerate(segments)}

for col_idx, (x_col, y_col, title) in enumerate(pairs, start=1):
    for seg in segments:
        subset = rfm_plot[rfm_plot['Segment'] == seg]
        fig.add_trace(
            go.Scatter(
                x=subset[x_col], y=subset[y_col],
                mode='markers',
                marker=dict(size=4, color=color_map[seg], opacity=0.5),
                name=seg,
                legendgroup=seg,
                showlegend=(col_idx == 1),
            ),
            row=1, col=col_idx,
        )

fig.update_layout(
    title_text='Customer Segments — 2D Projections',
    height=420,
    template='plotly_white',
)
fig.show()

**Business Insight — What the cluster visualisation shows:**

- The **Champions** cluster occupies the **low-Recency, high-Frequency, high-Monetary** corner of the space — exactly where we want our best customers to be. They are tightly grouped, confirming a coherent behavioral pattern.

- The **Dormant / At-Risk** cluster occupies the **high-Recency, low-Frequency, low-Monetary** region. These customers have not purchased in months and bought infrequently when they were active.

- The two clusters are **clearly separated** in all three 2D projections, particularly on the Recency axis. This validates the segmentation — we are not drawing arbitrary boundaries.

---
## Section 5 — Segment Profiles

Moving from visual clusters to business profiles: what do the numbers actually say about each group?

In [ ]:
# Clean segment summary table

segment_summary = (
    rfm.groupby('Segment')
    .agg(
        Customers     =('Recency',    'count'),
        Avg_Recency   =('Recency',    'mean'),
        Avg_Frequency =('Frequency',  'mean'),
        Avg_Monetary  =('Monetary',   'mean'),
        Total_Revenue =('Monetary',   'sum'),
    )
    .round(1)
)
segment_summary['Pct_Customers'] = (
    segment_summary['Customers'] / segment_summary['Customers'].sum() * 100
).round(1)
segment_summary['Pct_Revenue'] = (
    segment_summary['Total_Revenue'] / segment_summary['Total_Revenue'].sum() * 100
).round(1)
segment_summary = segment_summary.sort_values('Avg_Monetary', ascending=False)

print('=== Segment Profile Table ===')
print(segment_summary.to_string())

In [ ]:
# Bar charts: mean R, F, M per segment

metrics = ['Avg_Recency', 'Avg_Frequency', 'Avg_Monetary']
metric_labels = ['Avg Recency (days)', 'Avg Frequency (orders)', 'Avg Monetary (£)']
bar_colors = ['steelblue', 'coral', 'mediumseagreen']

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=metric_labels,
)

for col_idx, (metric, color) in enumerate(zip(metrics, bar_colors), start=1):
    data = segment_summary[metric].reset_index()
    fig.add_trace(
        go.Bar(
            x=data['Segment'],
            y=data[metric],
            marker_color=color,
            showlegend=False,
        ),
        row=1, col=col_idx,
    )

fig.update_layout(
    title_text='Mean RFM Values by Segment',
    height=420,
    template='plotly_white',
)
fig.show()

In [ ]:
# Revenue share donut chart

fig = px.pie(
    segment_summary.reset_index(),
    names='Segment',
    values='Total_Revenue',
    hole=0.45,
    title='Revenue Share by Segment',
    color_discrete_sequence=px.colors.qualitative.Bold,
    template='plotly_white',
)
fig.update_traces(textinfo='label+percent')
fig.update_layout(height=420)
fig.show()

**Business Insight — Segment Profiles in Plain English:**

---

### Champions (≈ 39% of customers)
Average recency **~50 days**, average **12.8 orders**, average spend **£6,569**.

Champions are the backbone of this business. They purchased recently, buy frequently, and spend substantially more than the average customer. While they represent only ~39% of the customer base, they account for a significantly **disproportionate share of total revenue** (see donut chart). These are the customers who would sign up for a loyalty programme, respond to early-access offers, and recommend the brand to others. The priority here is **retention and deepening engagement** — not acquisition.

---

### Dormant / At-Risk (≈ 61% of customers)
Average recency **~299 days** (roughly 10 months since last purchase), average **2.1 orders**, average spend **£613**.

This is the largest segment by headcount but contributes a much smaller share of revenue per customer. These buyers were likely one- or two-time purchasers who never developed a habit of returning. Many purchased once, possibly during a seasonal event, and drifted away. This group represents the **highest churn risk** but also the **biggest re-engagement opportunity**: even converting 10–15% of this group back to active buyers would meaningfully grow revenue.

---
## Section 6 — Business Action Framework

Data science delivers value when it translates into decisions. The table below maps each customer segment to its **risk level**, **commercial priority**, and **recommended actions**. This is the output that a data science team would present to marketing, CRM, and product stakeholders.

---

| Segment | Size | Revenue Share | Churn Risk | Recommended Actions |
|---|---|---|---|---|
| **Champions** | ~39% of customers | High (disproportionate) | Low | (1) Enrol in loyalty / VIP programme; (2) Give early access to new product launches; (3) Personalised product recommendations based on purchase history; (4) Thank-you emails and exclusive discounts to reinforce the relationship; (5) Use as a referral cohort — these customers are most likely to refer friends |
| **Dormant / At-Risk** | ~61% of customers | Low–Medium | High | (1) Win-back email campaign with time-limited discount (e.g. '15% off — we miss you'); (2) Segment further by recency sub-bucket: < 6 months = warm re-engagement; > 9 months = last-chance offer; (3) For customers with zero response after 2 touchpoints, reduce marketing spend (low ROI to continue pushing); (4) Investigate if Dormant customers have a common first-purchase category — personalise reactivation around that product |

---

### Why this segmentation matters — a quantified case

Assume we invest in a win-back campaign targeting the **3,566 Dormant / At-Risk** customers:

- Industry win-back conversion rates: **5–15%**
- Conservative assumption: **8% conversion** → **285 customers reactivated**
- If reactivated customers spend at even **50% of the Champion average** (£6,569 × 0.5 = £3,285)
- **Estimated incremental revenue: £934,000+**

This is the business case for Phase 3 (churn prediction): if we can identify *which* Dormant customers are most likely to respond, we can focus the campaign budget on the highest-ROI subset rather than blasting the entire 3,566-person group.

---

### Next steps (Phase 3 — Churn Prediction)

The segmentation produced here feeds directly into Phase 3:
- The **Dormant / At-Risk** segment provides the positive churn label
- RFM features + transaction-level behavioural features become the input to XGBoost/LightGBM
- The model will output a **churn probability score per customer**, enabling prioritised, targeted outreach rather than mass campaigns